# Module 7 — ATCNet-3C + Unsupervised Target Entropy/Pseudo-Label Adaptation

Final candidate for the prepared 3-class BCI-IV-2a cache.

- Input: `(N, 22, 640)` at 160 Hz
- Classes: `left`, `right`, `feet`
- Outer evaluation: S01–S09 LOSO
- Two source seeds: 42, 123
- Strict result: target labels are not used until scoring
- Transductive result: unlabeled target EEG may be used for entropy/pseudo-label adaptation

The adaptation is deliberately conservative: the pretrained backbone is frozen and only the final classifier heads are adapted with high-confidence pseudo-labels and consistency/entropy losses.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / SEED / DEVICE
# ============================================================

import os
import gc
import copy
import time
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Device:", device)
print("Seed:", SEED)

Device: cpu
Seed: 42


In [2]:
# ============================================================
# CELL 2 — EXISTING HDF5 CACHE
# ============================================================

PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")
PROJECT_DIR = PROJECT_ROOT / "cross_dataset_mi_project"
CACHE_DIR = PROJECT_DIR / "cache"

RESULT_DIR = PROJECT_DIR / "results" / "module_7_atcnet_adapt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CACHE_PATH = (
    CACHE_DIR /
    "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

if not FINAL_CACHE_PATH.exists():
    candidates = sorted(CACHE_DIR.glob("*.h5"))
    preferred = [
        p for p in candidates
        if "160hz" in p.name.lower() or "module_5" in p.name.lower()
    ]
    if not preferred:
        raise FileNotFoundError(
            f"No HDF5 cache found in {CACHE_DIR}"
        )
    FINAL_CACHE_PATH = preferred[0]

print("Cache:", FINAL_CACHE_PATH)
assert FINAL_CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — METADATA
# ============================================================

def decode(v):
    return v.decode("utf-8") if isinstance(v, bytes) else str(v)

with h5py.File(FINAL_CACHE_PATH, "r") as h5:
    X_shape = tuple(h5["X"].shape)
    X_dtype = str(h5["X"].dtype)
    metadata = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:
        metadata[key] = [
            decode(v) for v in h5["metadata"][key][:]
        ]

cache_meta_df = pd.DataFrame(metadata)
cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(len(cache_meta_df), dtype=np.int64),
)

CLASSES = ["left", "right", "feet"]
CLASS_TO_ID = {c:i for i,c in enumerate(CLASSES)}
ID_TO_CLASS = {i:c for c,i in CLASS_TO_ID.items()}
N_CLASSES = 3

assert X_shape[1:] == (22, 640)
assert X_dtype == "float32"

bci_meta = cache_meta_df[
    cache_meta_df["dataset"].astype(str) == "BCI-IV-2a"
].copy()

bci_meta["subject"] = bci_meta["subject"].astype(str)

BCI_SUBJECTS = sorted(
    bci_meta["subject"].unique()
)

assert len(BCI_SUBJECTS) == 9

print("Cache shape:", X_shape)
print("BCI subjects:", BCI_SUBJECTS)
print("BCI epochs:", len(bci_meta))

Cache shape: (9316, 22, 640)
BCI subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
BCI epochs: 1944


In [4]:
# ============================================================
# CELL 4 — HDF5 LOAD + QA
# ============================================================

def load_indices(indices):
    indices = np.asarray(indices, dtype=np.int64)
    with h5py.File(FINAL_CACHE_PATH, "r") as h5:
        return np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )

qa = load_indices(
    np.arange(min(512, X_shape[0]), dtype=np.int64)
)

print("Non-finite:", int((~np.isfinite(qa)).sum()))
print(
    "Zero variance:",
    int((np.var(qa, axis=(1,2)) <= 1e-12).sum())
)

assert np.isfinite(qa).all()
print("✅ QA PASS")

Non-finite: 0
Zero variance: 0
✅ QA PASS


In [5]:
# ============================================================
# CELL 5 — SOURCE-ONLY ROBUST NORMALIZER
# ============================================================

class SourceOnlyRobustNormalizer:

    def __init__(self, eps=1e-6):
        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=np.float32)

        V = (
            X.transpose(1,0,2)
            .reshape(X.shape[1], -1)
        )

        self.median_ = np.median(V, axis=1)

        q25 = np.percentile(V, 25, axis=1)
        q75 = np.percentile(V, 75, axis=1)

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Normalizer is not fitted.")

        X = np.asarray(X, dtype=np.float32)

        Z = (
            X
            - self.median_[None,:,None]
        ) / (
            self.iqr_[None,:,None]
            + self.eps
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(np.float32)

In [6]:
# ============================================================
# CELL 6 — STRATIFIED SOURCE VALIDATION
# ============================================================

def stratified_split(
    y,
    val_fraction=0.20,
    seed=SEED,
):
    y = np.asarray(y, dtype=np.int64)
    rng = np.random.default_rng(seed)

    all_idx = np.arange(len(y), dtype=np.int64)

    train_parts = []
    val_parts = []

    for cls in range(N_CLASSES):

        idx = all_idx[y == cls].copy()
        rng.shuffle(idx)

        n_val = max(
            1,
            int(round(len(idx) * val_fraction)),
        )

        val_parts.append(idx[:n_val])
        train_parts.append(idx[n_val:])

    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)

    return train_idx, val_idx

In [7]:
# ============================================================
# CELL 7 — ATCNET TCN
# ============================================================

class CausalConv1d(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation=1,
    ):
        super().__init__()

        self.pad = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=self.pad,
            dilation=dilation,
            bias=False,
        )

    def forward(self, x):
        y = self.conv(x)
        if self.pad:
            y = y[..., :-self.pad]
        return y


class TCNResidualBlock(nn.Module):

    def __init__(
        self,
        dim,
        filters=32,
        depth=2,
        kernel_size=4,
        dropout=0.30,
    ):
        super().__init__()

        self.proj = (
            nn.Conv1d(dim, filters, 1)
            if dim != filters
            else nn.Identity()
        )

        self.blocks = nn.ModuleList()

        for i in range(depth):
            dilation = 2 ** i

            self.blocks.append(
                nn.ModuleDict({
                    "c1": CausalConv1d(
                        filters,
                        filters,
                        kernel_size,
                        dilation,
                    ),
                    "bn1": nn.BatchNorm1d(filters),
                    "c2": CausalConv1d(
                        filters,
                        filters,
                        kernel_size,
                        dilation,
                    ),
                    "bn2": nn.BatchNorm1d(filters),
                    "drop": nn.Dropout(dropout),
                })
            )

    def forward(self, x):
        z = x.transpose(1,2)
        residual = self.proj(z)

        for block in self.blocks:

            h = block["c1"](residual)
            h = F.elu(block["bn1"](h))
            h = block["drop"](h)

            h = block["c2"](h)
            h = F.elu(block["bn2"](h))
            h = block["drop"](h)

            residual = F.elu(
                residual + h
            )

        return residual.transpose(1,2)

In [10]:
# ============================================================
# CELL 8 — CORRECTED ATCNET CONVOLUTIONAL BLOCK
# ============================================================

class ATCNetConvBlock(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        F1=16,
        D=2,
        kernel_size=64,
        pool1=8,
        pool2=7,
        dropout=0.30,
    ):

        super().__init__()

        F2 = F1 * D

        self.F2 = F2

        # ----------------------------------------------------
        # TEMPORAL CONVOLUTION
        #
        # Input:
        #   B x 1 x Channels x Time
        #
        # Kernel:
        #   1 x kernel_size
        #
        # Therefore this really operates along TIME.
        # ----------------------------------------------------

        self.temporal = nn.Conv2d(
            in_channels=1,
            out_channels=F1,
            kernel_size=(
                1,
                kernel_size,
            ),
            padding=(
                0,
                kernel_size // 2,
            ),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1
        )

        # ----------------------------------------------------
        # DEPTHWISE SPATIAL CONVOLUTION
        #
        # Operates across all 22 EEG channels.
        # ----------------------------------------------------

        self.spatial = nn.Conv2d(
            in_channels=F1,
            out_channels=F2,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F2
        )

        # ----------------------------------------------------
        # TEMPORAL POOLING
        #
        # Only pool along the TIME dimension.
        # ----------------------------------------------------

        self.pool1 = nn.AvgPool2d(
            kernel_size=(
                1,
                pool1,
            ),
            stride=(
                1,
                pool1,
            ),
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        # ----------------------------------------------------
        # TEMPORAL REFINEMENT
        # ----------------------------------------------------

        refine_kernel = 16

        self.refine = nn.Conv2d(
            in_channels=F2,
            out_channels=F2,
            kernel_size=(
                1,
                refine_kernel,
            ),
            padding=(
                0,
                refine_kernel // 2,
            ),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2
        )

        # ----------------------------------------------------
        # SECOND TEMPORAL POOL
        # ----------------------------------------------------

        self.pool2 = nn.AvgPool2d(
            kernel_size=(
                1,
                pool2,
            ),
            stride=(
                1,
                pool2,
            ),
        )

        self.drop2 = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):
        """
        Input:
            x = (B, C, T)

        Output:
            z = (B, T', F2)
        """

        # ----------------------------------------------------
        # B,C,T -> B,1,C,T
        # ----------------------------------------------------

        z = x.unsqueeze(
            1
        )

        # ----------------------------------------------------
        # Temporal feature extraction
        # ----------------------------------------------------

        z = self.temporal(
            z
        )

        z = self.bn1(
            z
        )

        z = F.elu(
            z
        )

        # ----------------------------------------------------
        # Spatial/depthwise EEG filtering
        #
        # Height becomes 1 here.
        # ----------------------------------------------------

        z = self.spatial(
            z
        )

        z = self.bn2(
            z
        )

        z = F.elu(
            z
        )

        # ----------------------------------------------------
        # Temporal pooling
        # ----------------------------------------------------

        z = self.pool1(
            z
        )

        z = self.drop1(
            z
        )

        # ----------------------------------------------------
        # Temporal refinement
        # ----------------------------------------------------

        z = self.refine(
            z
        )

        z = self.bn3(
            z
        )

        z = F.elu(
            z
        )

        # ----------------------------------------------------
        # Second temporal pooling
        # ----------------------------------------------------

        z = self.pool2(
            z
        )

        z = self.drop2(
            z
        )

        # ----------------------------------------------------
        # B,F,1,T -> B,F,T
        # ----------------------------------------------------

        z = z.squeeze(
            2
        )

        # ----------------------------------------------------
        # B,F,T -> B,T,F
        # ----------------------------------------------------

        z = z.transpose(
            1,
            2,
        )

        return z


print(
    "✅ Corrected ATCNet convolutional block loaded."
)

✅ Corrected ATCNet convolutional block loaded.


In [11]:
# ============================================================
# CELL 9 — CORRECTED ATCNET-3C MODEL
# ============================================================

class ATCNet3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_samples=640,
        n_classes=3,
        n_windows=5,
        F1=16,
        D=2,
        attn_heads=2,
        attn_dropout=0.30,
        tcn_depth=2,
        tcn_kernel=4,
        tcn_filters=32,
        tcn_dropout=0.30,
        dropout=0.30,
    ):

        super().__init__()

        self.n_windows = (
            n_windows
        )

        self.n_classes = (
            n_classes
        )

        self.conv = (
            ATCNetConvBlock(
                n_channels=n_channels,
                F1=F1,
                D=D,
                kernel_size=64,
                pool1=8,
                pool2=7,
                dropout=dropout,
            )
        )

        self.feature_dim = (
            F1 * D
        )

        # ----------------------------------------------------
        # Multi-head self-attention
        # ----------------------------------------------------

        self.attn = nn.ModuleList(
            [
                nn.MultiheadAttention(
                    embed_dim=self.feature_dim,
                    num_heads=attn_heads,
                    dropout=attn_dropout,
                    batch_first=True,
                )
                for _ in range(
                    n_windows
                )
            ]
        )

        self.attn_norm = nn.ModuleList(
            [
                nn.LayerNorm(
                    self.feature_dim
                )
                for _ in range(
                    n_windows
                )
            ]
        )

        # ----------------------------------------------------
        # TCN branch
        # ----------------------------------------------------

        self.tcn = nn.ModuleList(
            [
                TCNResidualBlock(
                    dim=self.feature_dim,
                    filters=tcn_filters,
                    depth=tcn_depth,
                    kernel_size=tcn_kernel,
                    dropout=tcn_dropout,
                )
                for _ in range(
                    n_windows
                )
            ]
        )

        # ----------------------------------------------------
        # One classifier per temporal window
        # ----------------------------------------------------

        self.window_head = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(
                        tcn_filters,
                        64,
                    ),

                    nn.ELU(),

                    nn.Dropout(
                        0.25
                    ),

                    nn.Linear(
                        64,
                        n_classes,
                    ),
                )
                for _ in range(
                    n_windows
                )
            ]
        )

        # ----------------------------------------------------
        # Determine compressed sequence length safely.
        # ----------------------------------------------------

        was_training = (
            self.conv.training
        )

        self.conv.eval()

        with torch.no_grad():

            dummy = torch.zeros(
                2,
                n_channels,
                n_samples,
            )

            seq = self.conv(
                dummy
            )

        if was_training:
            self.conv.train()

        self.seq_len = int(
            seq.shape[1]
        )

        if self.seq_len < (
            self.n_windows
        ):

            raise RuntimeError(
                "ATCNet compressed sequence "
                f"length={self.seq_len} is too short "
                f"for {self.n_windows} windows."
            )

    def forward(
        self,
        x,
    ):

        # ----------------------------------------------------
        # CNN feature extraction
        # ----------------------------------------------------

        z = self.conv(
            x
        )

        # z:
        # B,T,F

        logits = []

        # ----------------------------------------------------
        # Sliding temporal windows
        # ----------------------------------------------------

        for i in range(
            self.n_windows
        ):

            start = i

            end = (
                self.seq_len
                - self.n_windows
                + i
                + 1
            )

            w = z[
                :,
                start:end,
                :,
            ]

            # ------------------------------------------------
            # Attention
            # ------------------------------------------------

            a, _ = (
                self.attn[i](
                    w,
                    w,
                    w,
                    need_weights=False,
                )
            )

            w = self.attn_norm[i](
                w + a
            )

            # ------------------------------------------------
            # TCN
            # ------------------------------------------------

            w = self.tcn[i](
                w
            )

            # ------------------------------------------------
            # Last temporal representation
            # ------------------------------------------------

            last = w[
                :,
                -1,
                :,
            ]

            logits.append(
                self.window_head[i](
                    last
                )
            )

        # ----------------------------------------------------
        # Average temporal-window predictions
        # ----------------------------------------------------

        logits = torch.stack(
            logits,
            dim=0,
        )

        return logits.mean(
            dim=0
        )


# ============================================================
# FORWARD TEST
# ============================================================

_test_model = ATCNet3C().to(
    device
)

_test_model.eval()

with torch.no_grad():

    dummy_x = torch.randn(
        2,
        22,
        640,
        device=device,
    )

    dummy_out = _test_model(
        dummy_x
    )

print(
    "Compressed sequence length:",
    _test_model.seq_len,
)

print(
    "Output shape:",
    tuple(
        dummy_out.shape
    ),
)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in _test_model.parameters() if p.requires_grad):,}"
)

assert tuple(
    dummy_out.shape
) == (
    2,
    3,
)

print(
    "✅ ATCNet-3C forward test PASSED."
)

del _test_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Compressed sequence length: 11
Output shape: (2, 3)
Trainable parameters: 134,447
✅ ATCNet-3C forward test PASSED.


In [12]:
# ============================================================
# CELL 10 — TRAINING LOADERS + AUGMENTATION
# ============================================================

def augment_source_eeg(
    x,
):
    x = x.clone()

    B, C, T = x.shape

    if torch.rand(1, device=x.device).item() < 0.35:

        x = (
            x
            * torch.empty(
                B,1,1,
                device=x.device,
            ).uniform_(0.92,1.08)
        )

    if torch.rand(1, device=x.device).item() < 0.20:

        x = (
            x
            + 0.004
            * torch.randn_like(x)
        )

    return x


def make_train_loader(
    X,
    y,
    batch_size=64,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.from_numpy(y),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(np.float64)

    inv = np.zeros(
        N_CLASSES,
        dtype=np.float64,
    )

    valid = counts > 0
    inv[valid] = 1.0 / counts[valid]

    weights = inv[y]

    sampler = WeightedRandomSampler(
        torch.as_tensor(
            weights,
            dtype=torch.double,
        ),
        num_samples=len(y),
        replacement=True,
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

In [13]:
# ============================================================
# CELL 11 — PREDICTION + SOURCE TRAINING
# ============================================================

@torch.no_grad()
def predict_atcnet(
    model,
    X,
):

    model.eval()

    loader = make_eval_loader(X)

    outputs = []

    for xb, _ in loader:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        logits = model(xb)

        outputs.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        outputs,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    P = np.exp(logits)

    P /= (
        P.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return P.astype(
        np.float32
    )


def train_source_model(
    X_train,
    y_train,
    X_val,
    y_val,
    seed=42,
    epochs=150,
    batch_size=64,
    lr=9e-4,
    patience=30,
):

    seed_everything(seed)

    model = ATCNet3C().to(device)

    counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(np.float32)

    class_weights = (
        counts.sum()
        /
        (
            N_CLASSES
            * np.maximum(
                counts,
                1.0,
            )
        )
    )

    class_weights /= (
        class_weights.mean()
        + 1e-12
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=device,
        ),
        label_smoothing=0.01,
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.90,
            patience=12,
            min_lr=1e-5,
        )
    )

    loader = make_train_loader(
        X_train,
        y_train,
        batch_size=batch_size,
    )

    best_state = None
    best_loss = np.inf
    best_bacc = -np.inf
    best_epoch = 0
    wait = 0

    history = []

    print(
        "\n"
        + "-" * 72
    )

    print(
        f"ATCNet source seed = {seed}"
    )

    print(
        "-" * 72
    )

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        train_true = []
        train_pred = []
        losses = []

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            xb = augment_source_eeg(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            loss = criterion(
                logits,
                yb,
            )

            if not torch.isfinite(loss):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            losses.append(
                float(loss.item())
            )

            train_true.extend(
                yb.detach().cpu().numpy()
            )

            train_pred.extend(
                logits.argmax(1)
                .detach()
                .cpu()
                .numpy()
            )

        P_val = predict_atcnet(
            model,
            X_val,
        )

        pred_val = P_val.argmax(
            axis=1
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        train_acc = (
            accuracy_score(
                train_true,
                train_pred,
            ) * 100.0
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            ) * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            ) * 100.0
        )

        scheduler.step(
            val_loss
        )

        history.append({
            "epoch": epoch,
            "loss": float(
                np.mean(losses)
            ),
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "val_bacc": val_bacc,
            "lr": optimizer.param_groups[0]["lr"],
        })

        if val_loss < (
            best_loss - 1e-5
        ):

            best_loss = val_loss
            best_bacc = val_bacc
            best_epoch = epoch
            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"train={train_acc:5.1f}% | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

        if wait >= patience:

            print(
                f"    early stop at epoch "
                f"{epoch}; best={best_epoch}"
            )

            break

    if best_state is None:

        raise RuntimeError(
            "No valid source checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
        best_epoch,
        best_loss,
        best_bacc,
    )

In [14]:
# ============================================================
# CELL 12 — CONSERVATIVE TARGET ENTROPY + PSEUDO LABEL ADAPTATION
# ============================================================

def entropy_loss(
    probs,
):
    probs = torch.clamp(
        probs,
        1e-7,
        1.0,
    )

    return -(
        probs
        * torch.log(probs)
    ).sum(
        dim=1
    ).mean()


def make_target_view(
    x,
):

    z = x.clone()

    if torch.rand(
        1,
        device=z.device,
    ).item() < 0.50:

        z = (
            z
            * torch.empty(
                z.shape[0],
                1,
                1,
                device=z.device,
            ).uniform_(0.95,1.05)
        )

    if torch.rand(
        1,
        device=z.device,
    ).item() < 0.30:

        z = (
            z
            + 0.002
            * torch.randn_like(z)
        )

    return z


def adapt_target_model(
    source_model,
    X_target,
    X_val,
    y_val,
    epochs=20,
    batch_size=64,
    lr=1e-5,
    threshold=0.90,
):
    """
    Conservative transductive adaptation.

    Uses target EEG but never target labels.

    Backbone is frozen; only final ATCNet window heads
    are updated.
    """

    model = copy.deepcopy(
        source_model
    ).to(device)

    # Freeze everything.
    for p in model.parameters():
        p.requires_grad = False

    # Adapt only classification heads.
    for head in model.window_head:
        for p in head.parameters():
            p.requires_grad = True

    trainable = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=1e-4,
    )

    loader = make_eval_loader(
        X_target,
        batch_size=batch_size,
    )

    # Source validation is the only checkpoint criterion.
    P0 = predict_atcnet(
        model,
        X_val,
    )

    p0 = P0.argmax(
        axis=1
    )

    best_source_bacc = (
        balanced_accuracy_score(
            y_val,
            p0,
        ) * 100.0
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        # Keep BN/dropout stable.
        for module in model.modules():

            if isinstance(
                module,
                (
                    nn.BatchNorm1d,
                    nn.BatchNorm2d,
                ),
            ):
                module.eval()

            if isinstance(
                module,
                nn.Dropout,
            ):
                module.eval()

        pseudo_fracs = []
        e_losses = []
        p_losses = []
        c_losses = []

        for xb, _ in loader:

            xb = xb.to(
                device
            )

            x_view = make_target_view(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            # ------------------------------------------------
            # Teacher.
            # ------------------------------------------------

            with torch.no_grad():

                teacher_logits = model(
                    xb
                )

                teacher_prob = torch.softmax(
                    teacher_logits,
                    dim=1,
                )

                confidence, pseudo = (
                    teacher_prob.max(
                        dim=1
                    )
                )

                mask = (
                    confidence >= threshold
                )

            # ------------------------------------------------
            # Student.
            # ------------------------------------------------

            student_logits = model(
                x_view
            )

            student_prob = torch.softmax(
                student_logits,
                dim=1,
            )

            loss_ent = entropy_loss(
                student_prob
            )

            if mask.sum() >= 2:

                loss_pseudo = F.cross_entropy(
                    student_logits[mask],
                    pseudo[mask],
                )

            else:

                loss_pseudo = torch.zeros(
                    (),
                    device=device,
                )

            loss_consistency = F.mse_loss(
                student_prob,
                teacher_prob.detach(),
            )

            # Small weights to avoid collapse.
            loss = (
                0.03 * loss_ent
                +
                0.10 * loss_pseudo
                +
                0.05 * loss_consistency
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable,
                max_norm=0.10,
            )

            optimizer.step()

            pseudo_fracs.append(
                float(
                    mask.float()
                    .mean()
                    .item()
                )
            )

            e_losses.append(
                float(
                    loss_ent.item()
                )
            )

            p_losses.append(
                float(
                    loss_pseudo.item()
                )
            )

            c_losses.append(
                float(
                    loss_consistency.item()
                )
            )

        # ----------------------------------------------------
        # Source validation after adaptation.
        # ----------------------------------------------------

        P_val = predict_atcnet(
            model,
            X_val,
        )

        pred_val = P_val.argmax(
            axis=1
        )

        source_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            ) * 100.0
        )

        # Only accept adaptation if source validation
        # does not degrade by more than 1 percentage point.
        if source_bacc >= (
            best_source_bacc - 1.0
        ):

            if source_bacc > (
                best_source_bacc
            ):

                best_source_bacc = (
                    source_bacc
                )

                best_state = copy.deepcopy(
                    model.state_dict()
                )

        history.append({
            "epoch": epoch,
            "source_val_bacc": source_bacc,
            "pseudo_fraction": (
                np.mean(pseudo_fracs)
                if pseudo_fracs
                else 0.0
            ),
            "entropy": (
                np.mean(e_losses)
                if e_losses
                else np.nan
            ),
            "pseudo_loss": (
                np.mean(p_losses)
                if p_losses
                else np.nan
            ),
            "consistency": (
                np.mean(c_losses)
                if c_losses
                else np.nan
            ),
        })

        if (
            epoch == 1
            or epoch % 5 == 0
        ):

            print(
                f"      adapt {epoch:02d} | "
                f"src-bAcc={source_bacc:5.2f}% | "
                f"pseudo={100.0 * (np.mean(pseudo_fracs) if pseudo_fracs else 0.0):5.1f}%"
            )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
        best_source_bacc,
    )

In [15]:
# ============================================================
# CELL 13 — COMPLETE LOSO FOLD
# ============================================================

def run_loso_fold(
    target_subject,
    fold_id,
    seeds=(42,123),
    source_epochs=150,
    source_patience=30,
    adapt_epochs=20,
):
    t0 = time.time()

    target_subject = str(
        target_subject
    )

    source_mask = (
        bci_meta["subject"].astype(str)
        != target_subject
    )

    target_mask = (
        bci_meta["subject"].astype(str)
        == target_subject
    )

    source_idx = (
        bci_meta.loc[
            source_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_idx = (
        bci_meta.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_source_raw = load_indices(
        source_idx
    )

    X_target_raw = load_indices(
        target_idx
    )

    y_source = (
        bci_meta.loc[
            source_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_target = (
        bci_meta.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # --------------------------------------------------------
    # Source-only normalization.
    # --------------------------------------------------------

    normalizer = (
        SourceOnlyRobustNormalizer()
        .fit(
            X_source_raw
        )
    )

    X_source = normalizer.transform(
        X_source_raw
    )

    X_target = normalizer.transform(
        X_target_raw
    )

    # --------------------------------------------------------
    # Source validation.
    # --------------------------------------------------------

    train_idx, val_idx = (
        stratified_split(
            y_source,
            val_fraction=0.20,
            seed=SEED,
        )
    )

    X_train = X_source[
        train_idx
    ]

    y_train = y_source[
        train_idx
    ]

    X_val = X_source[
        val_idx
    ]

    y_val = y_source[
        val_idx
    ]

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"ATCNet + pseudo-label adaptation "
        f"LOSO [{fold_id}/9] — {target_subject}"
    )

    print(
        "=" * 78
    )

    print(
        "Source:",
        X_source.shape,
    )

    print(
        "Train:",
        X_train.shape,
    )

    print(
        "Val:",
        X_val.shape,
    )

    print(
        "Target:",
        X_target.shape,
    )

    # --------------------------------------------------------
    # Train seed models.
    # --------------------------------------------------------

    source_models = []
    seed_rows = []

    for seed in seeds:

        (
            model,
            history,
            best_epoch,
            best_loss,
            best_bacc,
        ) = train_source_model(
            X_train,
            y_train,
            X_val,
            y_val,
            seed=seed,
            epochs=source_epochs,
            batch_size=64,
            lr=9e-4,
            patience=source_patience,
        )

        source_models.append(
            model
        )

        seed_rows.append({
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_loss": best_loss,
            "best_val_bacc": best_bacc,
        })

    seed_summary = pd.DataFrame(
        seed_rows
    )

    print(
        "\nSeed summary:"
    )

    display(
        seed_summary
    )

    # --------------------------------------------------------
    # STRICT ENSEMBLE.
    # --------------------------------------------------------

    P_strict = np.mean(
        np.stack(
            [
                predict_atcnet(
                    model,
                    X_target,
                )
                for model in source_models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_strict = P_strict.argmax(
        axis=1
    )

    strict_acc = (
        accuracy_score(
            y_target,
            pred_strict,
        )
        * 100.0
    )

    strict_bacc = (
        balanced_accuracy_score(
            y_target,
            pred_strict,
        )
        * 100.0
    )

    strict_kappa = (
        cohen_kappa_score(
            y_target,
            pred_strict,
        )
    )

    # --------------------------------------------------------
    # TRANSDUCTIVE ADAPTATION.
    # --------------------------------------------------------

    adapted_models = []

    for model in source_models:

        adapted, adapt_hist, src_bacc = (
            adapt_target_model(
                model,
                X_target,
                X_val,
                y_val,
                epochs=adapt_epochs,
                batch_size=64,
                lr=1e-5,
                threshold=0.90,
            )
        )

        adapted_models.append(
            adapted
        )

    P_adapted = np.mean(
        np.stack(
            [
                predict_atcnet(
                    model,
                    X_target,
                )
                for model in adapted_models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_adapted = P_adapted.argmax(
        axis=1
    )

    adapted_acc = (
        accuracy_score(
            y_target,
            pred_adapted,
        )
        * 100.0
    )

    adapted_bacc = (
        balanced_accuracy_score(
            y_target,
            pred_adapted,
        )
        * 100.0
    )

    adapted_kappa = (
        cohen_kappa_score(
            y_target,
            pred_adapted,
        )
    )

    elapsed = time.time() - t0

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Target subject       : "
        f"{target_subject}"
    )

    print(
        f"Strict ensemble      : "
        f"{strict_acc:.2f}%"
    )

    print(
        f"Strict bAcc          : "
        f"{strict_bacc:.2f}%"
    )

    print(
        f"Adapted ensemble     : "
        f"{adapted_acc:.2f}%"
    )

    print(
        f"Adapted bAcc         : "
        f"{adapted_bacc:.2f}%"
    )

    print(
        f"Adapted kappa        : "
        f"{adapted_kappa:.4f}"
    )

    print(
        f"Elapsed              : "
        f"{elapsed / 60.0:.1f} min"
    )

    print(
        "-" * 78
    )

    return {
        "fold": fold_id,
        "subject": target_subject,
        "strict_acc": strict_acc,
        "strict_bacc": strict_bacc,
        "strict_kappa": strict_kappa,
        "adapted_acc": adapted_acc,
        "adapted_bacc": adapted_bacc,
        "adapted_kappa": adapted_kappa,
        "y_test": y_target,
        "strict_pred": pred_strict,
        "adapted_pred": pred_adapted,
        "P_strict": P_strict,
        "P_adapted": P_adapted,
        "seed_summary": seed_summary,
    }

In [16]:
# ============================================================
# CELL 14 — S01 SMOKE TEST
# ============================================================

smoke = run_loso_fold(
    target_subject="S01",
    fold_id=1,
    seeds=(42,123),
    source_epochs=120,
    source_patience=25,
    adapt_epochs=20,
)

print(
    "\n"
    + "=" * 78
)

print(
    "S01 RESULT"
)

print(
    "=" * 78
)

print(
    f"Strict accuracy  : "
    f"{smoke['strict_acc']:.2f}%"
)

print(
    f"Adapted accuracy : "
    f"{smoke['adapted_acc']:.2f}%"
)


ATCNet + pseudo-label adaptation LOSO [1/9] — S01
Source: (1728, 22, 640)
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)

------------------------------------------------------------------------
ATCNet source seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 36.7% | val= 36.2% | bAcc= 36.2% | vLoss=1.0709 | lr=9.00e-04
    epoch 010 | train= 65.9% | val= 60.9% | bAcc= 60.9% | vLoss=0.8098 | lr=9.00e-04
    epoch 020 | train= 76.1% | val= 67.0% | bAcc= 67.0% | vLoss=0.7706 | lr=9.00e-04
    epoch 030 | train= 81.6% | val= 65.8% | bAcc= 65.8% | vLoss=0.7922 | lr=8.10e-04
    epoch 040 | train= 86.4% | val= 66.1% | bAcc= 66.1% | vLoss=0.9279 | lr=7.29e-04
    epoch 050 | train= 89.4% | val= 70.1% | bAcc= 70.1% | vLoss=0.8320 | lr=7.29e-04
    early stop at epoch 51; best=26

------------------------------------------------------------------------
ATCNet source seed = 123
-------------------------------------------

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,26,0.742525,66.956522
1,123,15,0.725572,68.695652


      adapt 01 | src-bAcc=66.96% | pseudo= 23.4%
      adapt 05 | src-bAcc=66.67% | pseudo= 24.9%
      adapt 10 | src-bAcc=66.67% | pseudo= 23.8%
      adapt 15 | src-bAcc=66.67% | pseudo= 25.7%
      adapt 20 | src-bAcc=66.96% | pseudo= 25.3%
      adapt 01 | src-bAcc=68.70% | pseudo= 23.0%
      adapt 05 | src-bAcc=68.41% | pseudo= 22.0%
      adapt 10 | src-bAcc=68.41% | pseudo= 23.4%
      adapt 15 | src-bAcc=68.41% | pseudo= 23.4%
      adapt 20 | src-bAcc=67.83% | pseudo= 24.2%

------------------------------------------------------------------------------
Target subject       : S01
Strict ensemble      : 70.37%
Strict bAcc          : 70.37%
Adapted ensemble     : 70.37%
Adapted bAcc         : 70.37%
Adapted kappa        : 0.5556
Elapsed              : 8.5 min
------------------------------------------------------------------------------

S01 RESULT
Strict accuracy  : 70.37%
Adapted accuracy : 70.37%


In [17]:
# ============================================================
# CELL 15 — FULL 9-SUBJECT LOSO
# ============================================================

RUN_FULL_LOSO =True

if RUN_FULL_LOSO:

    results = []
    fold_objects = {}

    for fold_id, subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        result = run_loso_fold(
            target_subject=subject,
            fold_id=fold_id,
            seeds=(42,123),
            source_epochs=150,
            source_patience=30,
            adapt_epochs=25,
        )

        fold_objects[
            subject
        ] = result

        results.append({
            "fold": fold_id,
            "subject": subject,
            "strict_accuracy":
                result["strict_acc"],
            "strict_bacc":
                result["strict_bacc"],
            "adapted_accuracy":
                result["adapted_acc"],
            "adapted_bacc":
                result["adapted_bacc"],
            "adapted_kappa":
                result["adapted_kappa"],
        })

        gc.collect()

    results_df = pd.DataFrame(
        results
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "FINAL ATCNet-3C LOSO"
    )

    print(
        "=" * 78
    )

    display(
        results_df
    )

    print(
        "\nStrict mean:",
        f"{results_df['strict_accuracy'].mean():.2f}%"
    )

    print(
        "Adapted mean:",
        f"{results_df['adapted_accuracy'].mean():.2f}%"
    )

    print(
        "Adapted mean bAcc:",
        f"{results_df['adapted_bacc'].mean():.2f}%"
    )

    print(
        "Adapted median:",
        f"{results_df['adapted_accuracy'].median():.2f}%"
    )

    print(
        "Adapted std:",
        f"{results_df['adapted_accuracy'].std():.2f}%"
    )

    print(
        "Subjects >=70:",
        int(
            (
                results_df["adapted_accuracy"]
                >= 70.0
            ).sum()
        ),
        "/9",
    )

    mean_acc = results_df[
        "adapted_accuracy"
    ].mean()

    if mean_acc >= 80.0:
        print(
            "\n✅ 80% TARGET ACHIEVED."
        )
    elif mean_acc >= 70.0:
        print(
            "\n✅ 70% TARGET ACHIEVED."
        )
    else:
        print(
            "\n❌ TARGET NOT YET ACHIEVED."
        )


ATCNet + pseudo-label adaptation LOSO [1/9] — S01
Source: (1728, 22, 640)
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)

------------------------------------------------------------------------
ATCNet source seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 36.7% | val= 36.2% | bAcc= 36.2% | vLoss=1.0709 | lr=9.00e-04
    epoch 010 | train= 65.9% | val= 60.9% | bAcc= 60.9% | vLoss=0.8098 | lr=9.00e-04
    epoch 020 | train= 76.1% | val= 67.0% | bAcc= 67.0% | vLoss=0.7706 | lr=9.00e-04
    epoch 030 | train= 81.6% | val= 65.8% | bAcc= 65.8% | vLoss=0.7922 | lr=8.10e-04
    epoch 040 | train= 86.4% | val= 66.1% | bAcc= 66.1% | vLoss=0.9279 | lr=7.29e-04
    epoch 050 | train= 89.4% | val= 70.1% | bAcc= 70.1% | vLoss=0.8320 | lr=7.29e-04
    early stop at epoch 56; best=26

------------------------------------------------------------------------
ATCNet source seed = 123
-------------------------------------------

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,26,0.742525,66.956522
1,123,15,0.725572,68.695652


      adapt 01 | src-bAcc=66.96% | pseudo= 23.7%
      adapt 05 | src-bAcc=66.67% | pseudo= 24.1%
      adapt 10 | src-bAcc=66.67% | pseudo= 24.9%
      adapt 15 | src-bAcc=66.67% | pseudo= 25.3%
      adapt 20 | src-bAcc=66.96% | pseudo= 26.4%
      adapt 25 | src-bAcc=66.96% | pseudo= 25.3%
      adapt 01 | src-bAcc=68.70% | pseudo= 22.8%
      adapt 05 | src-bAcc=68.41% | pseudo= 22.0%
      adapt 10 | src-bAcc=68.41% | pseudo= 23.2%
      adapt 15 | src-bAcc=68.41% | pseudo= 23.6%
      adapt 20 | src-bAcc=67.83% | pseudo= 24.6%
      adapt 25 | src-bAcc=68.12% | pseudo= 23.6%

------------------------------------------------------------------------------
Target subject       : S01
Strict ensemble      : 70.37%
Strict bAcc          : 70.37%
Adapted ensemble     : 70.37%
Adapted bAcc         : 70.37%
Adapted kappa        : 0.5556
Elapsed              : 9.9 min
------------------------------------------------------------------------------

ATCNet + pseudo-label adaptation LOSO [2/9] 

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,20,0.660516,74.492754
1,123,16,0.638489,71.304348


      adapt 01 | src-bAcc=74.49% | pseudo=  7.4%
      adapt 05 | src-bAcc=74.49% | pseudo=  8.1%
      adapt 10 | src-bAcc=74.49% | pseudo=  9.6%
      adapt 15 | src-bAcc=74.49% | pseudo=  9.6%
      adapt 20 | src-bAcc=74.20% | pseudo= 10.0%
      adapt 25 | src-bAcc=74.20% | pseudo= 10.8%
      adapt 01 | src-bAcc=71.30% | pseudo=  6.8%
      adapt 05 | src-bAcc=71.30% | pseudo=  6.8%
      adapt 10 | src-bAcc=71.30% | pseudo=  6.8%
      adapt 15 | src-bAcc=71.30% | pseudo=  6.8%
      adapt 20 | src-bAcc=71.59% | pseudo=  7.6%
      adapt 25 | src-bAcc=71.59% | pseudo=  7.6%

------------------------------------------------------------------------------
Target subject       : S02
Strict ensemble      : 37.96%
Strict bAcc          : 37.96%
Adapted ensemble     : 38.43%
Adapted bAcc         : 38.43%
Adapted kappa        : 0.0764
Elapsed              : 9.7 min
------------------------------------------------------------------------------

ATCNet + pseudo-label adaptation LOSO [3/9] 

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 16 — CONFUSION MATRIX + SAVE
# ============================================================

if (
    "fold_objects" in globals()
    and len(fold_objects) > 0
):

    y_true = np.concatenate([
        fold_objects[s]["y_test"]
        for s in BCI_SUBJECTS
    ])

    pred = np.concatenate([
        fold_objects[s]["adapted_pred"]
        for s in BCI_SUBJECTS
    ])

    cm = confusion_matrix(
        y_true,
        pred,
        labels=list(
            range(N_CLASSES)
        ),
        normalize="true",
    )

    display(
        pd.DataFrame(
            cm,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        classification_report(
            y_true,
            pred,
            target_names=CLASSES,
            digits=4,
        )
    )

    if "results_df" in globals():

        results_path = (
            RESULT_DIR
            / "atcnet_entropy_pseudolabel_loso.csv"
        )

        results_df.to_csv(
            results_path,
            index=False,
        )

        print(
            "Saved:",
            results_path,
        )